# Assign morphotopes to the tree

Assign morphotopes and noise from the extension countries to the level 7 of the existing tree to retain consistency of the taxonomy.

In [2]:
import geopandas as gpd
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from libpysal.graph import read_parquet
import numpy as np

In [14]:
v = "v10"

model_params = "_post_processing_v1"

clusters_dir = "/data/uscuni-ulce/processed_data/clusters/"
chars_dir = "/data/uscuni-ulce/processed_data/chars/"
regions_datadir = "/data/uscuni-ulce/"
graph_dir = "/data/uscuni-ulce/processed_data/neigh_graphs/"
morphotopes_dir = "/data/uscuni-ulce/processed_data/morphotopes/"

In [5]:
region_hulls = pd.concat([
    gpd.read_parquet(
        regions_datadir + "regions/" + f"{country}_regions_hull.parquet"
    ) for country in ["fr_sp_nl_be", "extension_2"]
])

In [8]:
cdata = pd.read_parquet(f"/data/uscuni-ulce/processed_data/clusters/cdata_{v}.pq")
cdata.index = cdata.index.astype(str)

In [33]:
region_id = 849307

In [37]:
chars = pd.read_parquet(f"{chars_dir}primary_chars_{region_id}.parquet")
chars = chars[chars.index >= 0]
chars

,sdbAre,sdbPer,sdbCoA,ssbCCo,ssbCor,ssbSqu,ssbERI,ssbElo,ssbCCM,ssbCCD,...,mibElo,mibERI,mibCCo,mibLAL,mibFR,mibSCo,micBAD,licBAD,misBAD,midBAD
0,3677.188230,318.298346,0.0,0.297790,7.0,12.973238,0.861491,0.363879,49.864259,17.002350,...,0.364957,0.863032,0.299415,125.588318,11.631349,11.631349,1443.362632,1193.567288,1704.682917,144.579269
1,4231.302836,322.122524,0.0,0.306441,4.0,0.699153,1.000878,0.257468,66.100949,0.195315,...,0.258623,1.001407,0.307847,132.792369,13.210035,13.210035,1837.519406,1061.610058,1704.682917,2978.150501
2,841.518503,138.718474,0.0,0.340571,4.0,0.629474,1.000842,0.291102,27.962197,0.082671,...,0.293725,1.002064,0.343742,56.289717,6.138808,6.138808,314.904290,428.337113,328.524521,336.114116
3,806.412908,133.490638,0.0,0.359000,4.0,0.640647,1.000899,0.310158,26.655123,0.084573,...,0.312848,1.002168,0.362243,53.679369,6.112446,6.112446,325.701344,397.010895,205.914551,180.384845
4,15617.929786,567.919926,0.0,0.488035,6.0,0.596963,0.880371,0.962114,91.378124,12.678186,...,0.962166,0.881061,0.488841,202.055891,27.571828,27.571828,4207.670380,1957.114122,691.310008,663.523063
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121574,82.753598,37.253521,0.0,0.561366,4.0,2.214103,1.001065,0.640640,6.731657,0.120505,...,0.646770,1.005527,0.570084,13.900153,2.283704,2.283704,591.899366,231.480125,45.809197,45.809197
121575,279.785529,85.022003,0.0,0.286368,6.0,24.026718,0.919997,0.317510,12.955753,6.316325,...,0.321365,0.924198,0.291784,35.469970,3.366263,3.366263,281.617710,176.808101,117.301032,126.191224
121576,184.321196,56.872019,0.0,0.527425,4.0,0.782403,1.000955,0.538575,10.487081,0.060013,...,0.543492,1.003903,0.533622,21.294159,3.305034,3.305034,1077.460976,913.081091,137.919523,137.141133
121577,84.622222,36.941215,0.0,0.626320,4.0,0.036847,1.000024,0.836876,6.555891,0.002075,...,0.840057,1.004550,0.634400,13.315923,2.351594,2.351594,28.033801,22.908619,35.603849,36.175612


In [35]:
morphotopes = pd.read_parquet(
        f"/data/uscuni-ulce/processed_data/morphotopes/tessellation_labels_morphotopes_{region_id}{model_params}.pq"
    )
morphotopes

,morphotope_label
focal,
0,0_62
1,0_62
2,1_12
3,1_10
4,2_19
...,...
121574,352_2
121575,7501_-1
121576,130_2


In [38]:
is_noise = morphotopes.morphotope_label.str.contains("-1")

In [41]:
# align morphotopes and cdata
morphotope_groups = chars[~is_noise].groupby(morphotopes[~is_noise].morphotope_label).median()
morphotope_chars = pd.read_parquet(f"/data/uscuni-ulce/processed_data/morphotopes/morph_chars_{region_id}.pq")
morphotope_groups["limAre"] = morphotope_chars['limAre']
morphotope_groups["limLPS"] = morphotope_chars['limLPS']
morphotope_groups = morphotope_groups[cdata.columns]

In [43]:
    # combine the cluster and the region morphotope chars
    # and scale them
    all_data = pd.concat((cdata, morphotope_groups))
    scalar = StandardScaler()
    vals = scalar.fit_transform(all_data)
    all_data = pd.DataFrame(
        vals, index=all_data.index, columns=all_data.columns
    ).fillna(0)
    
    # setup tree data
    morphotope_query_data = all_data.loc[morphotope_groups.index]
    tree_data = all_data.loc[cdata.index]
    
    ## query a knn classifier and assign predictions
    tree = KNeighborsClassifier(n_neighbors=1, n_jobs=-1)
    tree = tree.fit(X=tree_data.values, y=cdata.index.values.astype(str))
    morphotope_predictions = tree.predict(morphotope_query_data.values)


In [47]:
# assign morphotope labels
morphotopes.loc[~is_noise, 'final'] = morphotopes.loc[~is_noise, 'morphotope_label'].map(
    dict(zip(morphotope_groups.index, morphotope_predictions.astype(int)))
)

In [49]:
 # ========== noise ==========
with_noise = morphotopes.final.copy()


In [50]:
tq1 = read_parquet(graph_dir + f"tessellation_graph_{region_id}.parquet")
noise_graph = tq1.subgraph(morphotopes[is_noise].index)
components = noise_graph.component_labels
noise_groups = chars[is_noise].groupby(components).median()

In [51]:
# [noise] approax limAre
noise_groups["limAre"] = (
    chars[is_noise]
    .groupby(components)["sdbAre"]
    .apply(lambda x: x.sort_values(ascending=True)[-min(30, x.shape[0]) :].sum())
)
noise_groups = noise_groups[cdata.columns.drop("limLPS")]
noise_groups.index = "-1_" + noise_groups.index.astype(str)

all_data = pd.concat((cdata.drop("limLPS", axis=1), noise_groups))
scalar = StandardScaler()
vals = scalar.fit_transform(all_data)
all_data = pd.DataFrame(
    vals, index=all_data.index, columns=all_data.columns
).fillna(0)

# setup tree data
query_data = all_data.loc[noise_groups.index]
tree_data = all_data.loc[cdata.index]

## query a knn classifier and assign predictions
tree = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
tree = tree.fit(X=tree_data.values, y=cdata.index.values)
predictions = tree.predict(query_data.values)

# map noise group clusters to building ids
bid_to_ngroup = noise_graph.component_labels
bid_to_prediction = bid_to_ngroup.map(
    pd.Series(
        predictions, query_data.index.str.split("_").str[-1].astype(int)
    ).to_dict()
)
morphotopes.loc[bid_to_prediction.index.values, "final"] = (
    bid_to_prediction.values.astype(int)
)